# Semantic correspondence: Colab runner

This notebook sets options and calls the repo code (`scripts/colab_setup.py`, `run.py`).
Code changes belong in the repo: edits made here are lost. Full guide: `RUN.md`.

**Before the first run:**
1. Runtime > Change runtime type > **GPU** (needed for `train`; `evaluate`/`submit` also work on CPU, slowly).
2. Accept the competition rules on Kaggle.
3. Add to Colab Secrets (key icon) with Notebook access ON: `GITHUB_TOKEN`, `KAGGLE_KEY` (and `KAGGLE_USERNAME` only for legacy keys; `HF_TOKEN` / `WANDB_API_KEY` are optional).

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [13]:
# Run settings
BRANCH = "giuseppe"      # branch to run; push your changes there first
MODE = "train"           # "train" | "evaluate" | "submit"   (settings: BASE in src/config.py)
RUN = ""                 # evaluate/submit an existing run (uses its best.pt)
ENSEMBLE = ""            # comma-separated run IDs to ensemble ("" = single RUN)
RESUME = ""              # train: continue this run id after a disconnect
EXTRA = ""               # other run.py flags, e.g. "--dry-run"

# Unified preset & transformation switches (passed via CLI flags so src/config.py stays clean for git pull):
PRESET = "cleandift_pure"      # "cleandift_pure" (CleanDIFT SD2.1) | "cleandift_frozen" | "sd_pure" | "unified_lora_large" (0.70)
CROSS_VAL = False              # True = 5-fold image-disjoint Group K-Fold CV (trains CV_FOLDS when MODE='train')
CV_FOLDS = [0, 1, 2, 3, 4]     # Subset of folds to run when CROSS_VAL=True (e.g. [2, 3, 4] to continue later)
VAL_FOLD = None                # Ignored if CROSS_VAL=True; None = legacy split, 0..4 = single image-disjoint fold
CSLS_WEIGHT = None             # None = 0.0 (off)
OBJECT_ZOOM = True             # 2-stage object-centric zoom at inference (src/models/zoom.py)
CROSS_TRANSFORM = False        # src/models/cross_transform.py (off by default: overfits 314 pairs)
EVAL_IMAGE_SIZE = None         # e.g. 840 for higher-res evaluate/submit (None = use run's 644)
FUSION = "none"                # "none" (fast pure DINOv2) | "concat" | "adapter"
SUBMISSION_NAME = "submission_cleandift_pure.csv"  # filename saved on Drive & uploaded to Kaggle

In [14]:
# Clone the private repo (using GITHUB_TOKEN secret) and run scripts/colab_setup.py:
# pull, install environment.yml (pip part), secrets, Drive, data and weights.
import os
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN").strip()
    REPO_URL = f"https://{token}@github.com/julpfi/sapienza_aml_semantic_correspondence.git"
except Exception:
    REPO_URL = "https://github.com/julpfi/sapienza_aml_semantic_correspondence.git"

# Support new Kaggle API tokens saved in KAGGLE_KEY without KAGGLE_USERNAME
try:
    k_key = userdata.get("KAGGLE_KEY").strip()
    try:
        userdata.get("KAGGLE_USERNAME")
    except Exception:
        if k_key and not os.environ.get("KAGGLE_API_TOKEN"):
            os.environ["KAGGLE_API_TOKEN"] = k_key
except Exception:
    pass

REPO_DIR = "/content/sapienza_aml_semantic_correspondence"
if not os.path.isdir(f"{REPO_DIR}/.git"):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git checkout -- src/config.py 2>/dev/null || true
%run scripts/colab_setup.py --branch {BRANCH}

/content/sapienza_aml_semantic_correspondence
$ git fetch origin giuseppe
$ git checkout giuseppe
$ git pull --ff-only origin giuseppe
$ /usr/bin/python3 -m pip install -q transformers==5.22.1 huggingface_hub==2.4.0 diffusers==0.43.0 numpy==2.4.6 pandas==3.1.0 pillow==13.0.0 tqdm==4.70.1 matplotlib==3.11.2 wandb==0.31.0 kaggle==2.2.5 six==1.17.0 python-dotenv==1.2.4 pyyaml==6.0.3 ipykernel==7.4.0
package           environment.yml   installed
torch             2.15.0            2.14.1+cu130
torchvision       0.29.2            0.29.1+cu130
transformers      5.22.1            5.22.1
huggingface_hub   2.4.0             2.4.0
diffusers         0.43.0            0.43.0
numpy             2.4.6             2.4.6
pandas            3.1.0             3.1.0
pillow            13.0.0            13.0.0
tqdm              4.70.1            4.70.1
matplotlib        3.11.2            3.11.2
wandb             0.31.0            0.31.0
kaggle            2.2.5             2.2.5
six               1.17.0      

In [15]:
# Start the run. Runs are saved on your Drive: aml_semantic_correspondence/outputs/<run_id>
import json
from pathlib import Path
import numpy as np

def _build_flags(val_fold):
    f = f"--mode {MODE} {EXTRA}"
    if RESUME:
        f += f" --resume {RESUME}"
    elif ENSEMBLE and MODE != "train":
        f += f" --ensemble {ENSEMBLE}"
    elif RUN:
        f += f" --run {RUN}"
    elif PRESET and PRESET != "none":
        f += f" --preset {PRESET}"
    if MODE == "train":
        f += " --require-gpu"
        if not RESUME:
            f += " --cross-transform" if CROSS_TRANSFORM else " --no-cross-transform"
            f += f" --fusion {FUSION}"
    else:
        if EVAL_IMAGE_SIZE:
            f += f" --image-size {EVAL_IMAGE_SIZE}"
        if MODE == "submit" and SUBMISSION_NAME:
            f += f" --submission-name {SUBMISSION_NAME}"
    if val_fold is not None:
        f += f" --val-fold {val_fold}"
    if CSLS_WEIGHT is not None:
        f += f" --csls-weight {CSLS_WEIGHT}"
    f += " --object-zoom" if OBJECT_ZOOM else " --no-object-zoom"
    return f

if CROSS_VAL and MODE == "train":
    out_root = Path("/content/drive/MyDrive/aml_semantic_correspondence/outputs")
    cv_runs, cv_scores, cv_done = [], [], []
    for fold in CV_FOLDS:
        before = set(p.name for p in out_root.glob("*_train")) if out_root.is_dir() else set()
        flags = _build_flags(fold)
        print(f"\n{'='*60}\n[Group K-Fold CV] Fold {fold} ({CV_FOLDS}) | WANDB_MODE=disabled python run.py {flags}\n{'='*60}")
        !WANDB_MODE=disabled python run.py {flags}
        after = sorted(set(p.name for p in out_root.glob("*_train")) - before) if out_root.is_dir() else []
        if after:
            rid = after[-1]
            cv_runs.append(rid)
            cv_done.append(fold)
            hist_path = out_root / rid / "history.json"
            if hist_path.is_file():
                hist = json.loads(hist_path.read_text())
                best = max((h.get("best_pck", h.get("pck@0.1", 0.0)) for h in hist), default=0.0)
                cv_scores.append(best)
                print(f"-> Fold {fold} finished: {rid} | best val PCK@0.10 = {best:.2f}%")
    if cv_scores:
        mean_pck, std_pck = float(np.mean(cv_scores)), float(np.std(cv_scores))
        print(f"\n{'='*60}\nGroup K-Fold CV ({cv_done}) Mean PCK@0.10: {mean_pck:.2f}% ± {std_pck:.2f}%")
        print("Per-fold:", dict(zip(cv_done, [round(s, 2) for s in cv_scores])))
        print(f"Run IDs: ENSEMBLE = \"{','.join(cv_runs)}\"\n{'='*60}")
        if os.environ.get("WANDB_API_KEY"):
            import wandb
            from src.config import get_config
            _cfg = get_config(preset=None if PRESET in (None, "", "none") else PRESET)
            _wb = wandb.init(project=_cfg["wandb_project"], entity=_cfg["wandb_entity"],
                             name=f"cv_{PRESET or 'base'}_{''.join(map(str, cv_done))}", job_type="cv",
                             config={"preset": PRESET, "cv_folds": cv_done, "cv_runs": cv_runs, "object_zoom": OBJECT_ZOOM},
                             settings=wandb.Settings(x_disable_stats=True, console="off"))
            _wb.summary.update({"cv/mean_pck@0.1": mean_pck, "cv/std_pck@0.1": std_pck,
                                **{f"cv/fold_{f}_pck@0.1": s for f, s in zip(cv_done, cv_scores)}})
            _wb.finish()
else:
    flags = _build_flags(VAL_FOLD)
    print("Running: python run.py", flags)
    !python run.py {flags}

Running: python run.py --mode evaluate  --run 20261121-180216_train --object-zoom --constellation-align
Run 20261121-180216_train -> /content/drive/MyDrive/aml_semantic_correspondence/outputs/20261121-180216_train (settings in config.json)

Loaded weights from /content/drive/MyDrive/aml_semantic_correspondence/outputs/20261121-180216_train/checkpoints/best.pt
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (

{
  "n_queries": 453,
  "n_pairs": 78,
  "pck@0.05": 50.9934,
  "pck@0.1": 65.7837,
  "pck@0.15": 72.4062,
  "pck@0.1_ci95": 4.369,
  "score": 0.6578,
  "flip_rate": 0.0,
  "correct_pct": 65.7837,
  "other_miss_pct": 9.9338,
  "near_miss_pct": 9.4923,
  "wrong_keypoint_pct": 8.1678,
  "far_miss_pct": 6.6225,
  "pck@0.1/multi": 71.4876,
  "pck@0.1/single": 59.2417,
  "sec_per_query": 0.1571,
  "loss_over_chance": 0.3509,
  "

In [16]:
# Optional: paired comparison of evaluated runs (seeds comma-separated, A then B), e.g. "a_s0,a_s1 b_s0,b_s1".
# Each run needs per_query.csv: MODE = "evaluate" with RUN = "<run_id>" writes it.
COMPARE = ""
if COMPARE:
    !python scripts/compare_runs.py {COMPARE}

In [8]:
# Optional: upload a submission to Kaggle (set the run id printed above).
SUBMIT_RUN = ""             # set to the run id you want to submit, or "" to skip
SUBMIT_MESSAGE = ""         # optional Kaggle description (defaults to SUBMIT_RUN + filename)
if SUBMIT_RUN:
    sub_file = SUBMISSION_NAME if SUBMISSION_NAME.endswith(".csv") else f"{SUBMISSION_NAME}.csv"
    sub_msg = SUBMIT_MESSAGE or f"{SUBMIT_RUN} | {sub_file[:-4]}"
    !python scripts/kaggle_io.py submit --file /content/drive/MyDrive/aml_semantic_correspondence/outputs/{SUBMIT_RUN}/{sub_file} --message "{sub_msg}"


Submission ref: 58682359
2 submissions remaining today.
Successfully submitted to AML Challenge 2026/27Submitted. Check the score: kaggle competitions submissions -c aml-challenge-2026-27
